# 🚀 머신러닝 실습 : 고객 구매 데이터로 성별 예측 모델링 (분류 문제)
- 주어진 데이터는 백화점 고객의 1년 간 구매 데이터입니다.
- 고객 3,500명에 대한 학습용 데이터(y.csv, X.csv)를 이용하여 성별예측 모형을 만들어보세요.
- 모델의 성능은 자유롭게 측정해봅니다!
## [실습 프로세스]
1. 데이터 불러오기
2. 데이터 탐색
3. 데이터 전처리
4. 학습/테스트 데이터 분리
5. 모델 선택 및 학습
6. 예측 및 평가m

# 0. 라이브러리 불러오기
- 라이브러리를 가져와서 과정을 준비합니다

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings(action='ignore')

# 1. 데이터 불러오기
- 데이터를 가져와서 과정을 준비합시다.
- 인코딩 방식은 'euc-kr' 을 활용하세요.
- 데이터 출처 : 한국데이터산업진흥원 빅데이터분석기사 실기 공개 예시 문항
- 독립 변수 데이터셋 : ./03_ml_data/tabular_datasets/X.csv
- 종속 변수 데이터셋 : ./03_ml_data/tabular_datasets/y.csv

데이터 파일을 불러옵니다. 보통 CSV 파일을 pandas로 읽어옵니다.

In [2]:
from pathlib import Path

def find_repo_file(*parts):
    start = Path.cwd().resolve()
    for base in (start, *start.parents):
        candidate = base.joinpath(*parts)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(parts)

X_PATH = find_repo_file('05_machine_learning', '03_ml_data', 'tabular_datasets', '../03_ml_data/tabular_datasets/X.csv')
Y_PATH = find_repo_file('05_machine_learning', '03_ml_data', 'tabular_datasets', '../03_ml_data/tabular_datasets/y.csv')
print('X_PATH =', X_PATH)
print('Y_PATH =', Y_PATH)

현재 작업 폴더: C:\Users\Admin\hipython\ml


In [3]:
X = pd.read_csv(X_PATH, encoding='euc-kr')
y = pd.read_csv(Y_PATH, encoding='euc-kr')

In [4]:
X.head()

,cust_id,총구매액,최대구매액,환불금액,주구매상품,주구매지점,내점일수,내점당구매건수,주말방문비율,구매주기
0,0,68282840,11264000,6860000.0,기타,강남점,19,3.894737,0.527027,17
1,1,2136000,2136000,300000.0,스포츠,잠실점,2,1.500000,0.000000,1
2,2,3197000,1639000,NaN,남성 캐주얼,관악점,2,2.000000,0.000000,1
3,3,16077620,4935000,NaN,기타,광주점,18,2.444444,0.318182,16
4,4,29050000,24000000,NaN,보석,본 점,2,1.500000,0.000000,85


In [5]:
y.head()

,cust_id,gender
0,0,0
1,1,0
2,2,1
3,3,1
4,4,0


# 2. 데이터 탐색하기
- 데이터를 이해할 수 있도록 탐색과정을 수행해봅시다.
데이터의 상위 몇 개 행을 출력하여 전체 구조를 미리 확인합니다.

데이터의 요약 정보나 통계 정보를 출력해 변수들의 유형과 분포를 확인합니다.

데이터의 요약 정보나 통계 정보를 출력해 변수들의 유형과 분포를 확인합니다.

In [6]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   cust_id  3500 non-null   int64  
 1   총구매액     3500 non-null   int64  
 2   최대구매액    3500 non-null   int64  
 3   환불금액     1205 non-null   float64
 4   주구매상품    3500 non-null   str    
 5   주구매지점    3500 non-null   str    
 6   내점일수     3500 non-null   int64  
 7   내점당구매건수  3500 non-null   float64
 8   주말방문비율   3500 non-null   float64
 9   구매주기     3500 non-null   int64  
dtypes: float64(3), int64(5), str(2)
memory usage: 273.6 KB


In [7]:
y.info()

<class 'pandas.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   cust_id  3500 non-null   int64
 1   gender   3500 non-null   int64
dtypes: int64(2)
memory usage: 54.8 KB


In [8]:
X.describe()

,cust_id,총구매액,최대구매액,환불금액,내점일수,내점당구매건수,주말방문비율,구매주기
count,3500.000000,3.500000e+03,3.500000e+03,1.205000e+03,3500.000000,3500.000000,3500.000000,3500.000000
mean,1749.500000,9.191925e+07,1.966424e+07,2.407822e+07,19.253714,2.834963,0.307246,20.958286
std,1010.507298,1.635065e+08,3.199235e+07,4.746453e+07,27.174942,1.912368,0.289752,24.748682
min,0.000000,-5.242152e+07,-2.992000e+06,5.600000e+03,1.000000,1.000000,0.000000,0.000000
25%,874.750000,4.747050e+06,2.875000e+06,2.259000e+06,2.000000,1.666667,0.027291,4.000000
50%,1749.500000,2.822270e+07,9.837000e+06,7.392000e+06,8.000000,2.333333,0.256410,13.000000
75%,2624.250000,1.065079e+08,2.296250e+07,2.412000e+07,25.000000,3.375000,0.448980,28.000000
max,3499.000000,2.323180e+09,7.066290e+08,5.637530e+08,285.000000,22.083333,1.000000,166.000000


In [10]:
y.describe()

,cust_id,gender
count,3500.000000,3500.000000
mean,1749.500000,0.376000
std,1010.507298,0.484449
min,0.000000,0.000000
25%,874.750000,0.000000
50%,1749.500000,0.000000
75%,2624.250000,1.000000
max,3499.000000,1.000000


In [11]:
X.columns

Index(['cust_id', '총구매액', '최대구매액', '환불금액', '주구매상품', '주구매지점', '내점일수', '내점당구매건수',
       '주말방문비율', '구매주기'],
      dtype='str')

In [12]:
y.columns

Index(['cust_id', 'gender'], dtype='str')

In [13]:
print("X shape:", X.shape)
print("y shape:", y.shape)

# 키 중복/결측 체크
print("\n[Key 체크]")
print("X cust_id unique:", X["cust_id"].nunique(), " / rows:", len(X))
print("y cust_id unique:", y["cust_id"].nunique(), " / rows:", len(y))
print("X cust_id null:", X["cust_id"].isna().sum())
print("y cust_id null:", y["cust_id"].isna().sum())

# 병합 (1:1 가정)
df = pd.merge(X, y, on="cust_id", how="inner")

print("\nMerged df shape:", df.shape)
print("Merged cust_id unique:", df["cust_id"].nunique())

# -----------------------------
# 1) 상위/하위 행 확인
# -----------------------------
print("\n[Head]")
print(df.head())

print("\n[Tail]")
print(df.tail())

# -----------------------------
# 2) 데이터 구조/타입/결측 확인
# -----------------------------
print("\n[Info]")
df.info()

print("\n[결측치 개수(내림차순)]")
na_cnt = df.isna().sum().sort_values(ascending=False)
print(na_cnt[na_cnt > 0])

# -----------------------------
# 3) 타깃 분포 확인
# -----------------------------
print("\n[Target 분포: gender]")
print(df["gender"].value_counts(dropna=False))
print("\n[Target 비율]")
print(df["gender"].value_counts(normalize=True, dropna=False).round(4))

# -----------------------------
# 4) 수치형/범주형 컬럼 분리
# -----------------------------
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

# cust_id는 식별자이므로 분석용에서 제외 권장
if "cust_id" in num_cols:
    num_cols.remove("cust_id")
if "cust_id" in cat_cols:
    cat_cols.remove("cust_id")

# target 제외(설명/분포용으로는 별도)
if "gender" in num_cols:
    num_cols.remove("gender")
if "gender" in cat_cols:
    cat_cols.remove("gender")

print("\n[수치형 컬럼]", num_cols)
print("[범주형 컬럼]", cat_cols)

# -----------------------------
# 5) 수치형 요약 통계
# -----------------------------
print("\n[수치형 describe()]")
print(df[num_cols].describe().T)

# 이상치 감지용: IQR 기반 범위
print("\n[수치형 IQR 범위(간단 이상치 체크)]")
iqr_rows = []
for c in num_cols:
    q1 = df[c].quantile(0.25)
    q3 = df[c].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    out_cnt = ((df[c] < lower) | (df[c] > upper)).sum()
    iqr_rows.append((c, lower, upper, int(out_cnt)))
iqr_df = pd.DataFrame(iqr_rows, columns=["col", "lower_iqr", "upper_iqr", "outlier_cnt"])
print(iqr_df.sort_values("outlier_cnt", ascending=False))

# -----------------------------
# 6) 범주형 요약(상위 빈도)
# -----------------------------
print("\n[범주형 value_counts top5]")
for c in cat_cols:
    print(f"\n- {c}")
    print(df[c].value_counts(dropna=False).head(5))

# -----------------------------
# 7) 성별별(타깃별) 그룹 통계
# -----------------------------
print("\n[gender별 수치형 평균/중앙값]")
group_mean = df.groupby("gender")[num_cols].mean(numeric_only=True)
group_median = df.groupby("gender")[num_cols].median(numeric_only=True)

print("\nMean")
print(group_mean)

print("\nMedian")
print(group_median)

# -----------------------------
# 8) 상관관계(수치형)
# -----------------------------
print("\n[수치형 상관계수 상위(절대값 기준) - 중복 제거]")
corr = df[num_cols].corr(numeric_only=True).abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
top_corr = upper.stack().sort_values(ascending=False).head(15)
print(top_corr)

# -----------------------------
# 9) 데이터 품질 체크(중복행)
# -----------------------------
print("\n[중복행 체크]")
print("전체 중복행 수:", df.duplicated().sum())
print("cust_id 중복 수:", df["cust_id"].duplicated().sum())

X shape: (3500, 10)
y shape: (3500, 2)

[Key 체크]
X cust_id unique: 3500  / rows: 3500
y cust_id unique: 3500  / rows: 3500
X cust_id null: 0
y cust_id null: 0

Merged df shape: (3500, 11)
Merged cust_id unique: 3500

[Head]
   cust_id      총구매액     최대구매액       환불금액   주구매상품 주구매지점  내점일수   내점당구매건수  \
0        0  68282840  11264000  6860000.0      기타   강남점    19  3.894737   
1        1   2136000   2136000   300000.0     스포츠   잠실점     2  1.500000   
2        2   3197000   1639000        NaN  남성 캐주얼   관악점     2  2.000000   
3        3  16077620   4935000        NaN      기타   광주점    18  2.444444   
4        4  29050000  24000000        NaN      보석  본  점     2  1.500000   

     주말방문비율  구매주기  gender  
0  0.527027    17       0  
1  0.000000     1       0  
2  0.000000     1       1  
3  0.318182    16       1  
4  0.000000    85       0  

[Tail]
      cust_id       총구매액     최대구매액       환불금액 주구매상품 주구매지점  내점일수   내점당구매건수  \
3495     3495    3175200   3042900        NaN    골프  본  점     1  2.00000

# 3. 데이터 전처리
- 전처리 과정을 통해서 머신러닝에 사용할 수 있는 형태의 데이터 준비
필요한 라이브러리를 불러옵니다.

- 인코딩 : LabelEncoder
- 데이터 표준화 : StandardScaler
- 단순히 1부터의 숫자를 부여한 'cust_id'를 수치형 변수로 받아들이면, 결과가 왜곡될 수 있으니 컬럼을 제거합니다.
- 데이터에 결측치가 있는지 확인해보세요
- 결측치에 0으로 채워 넣어 모델 학습에 지장이 없도록 합니다.
문자형 범주 데이터를 숫자로 바꾸기 위한 인코딩을 수행합니다.
각 데이터에 표준화를 적용하여 데이터의 스케일(크기 차이)을 맞춰줍니다.
- 평균을 0, 표준편차를 1로 맞춰서 → 데이터가 정규 분포 형태로 변환되도록 하세요

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# -----------------------------
# 0) 데이터 로드 + 병합
# -----------------------------
X = pd.read_csv(X_PATH, encoding="cp949")
y = pd.read_csv(Y_PATH, encoding="cp949")

df = pd.merge(X, y, on="cust_id", how="inner")

print("Merged shape:", df.shape)
print(df.head())

# -----------------------------
# 1) cust_id 제거 (식별자)
# -----------------------------
df = df.drop(columns=["cust_id"])

# -----------------------------
# 2) 결측치 확인
# -----------------------------
na_cnt = df.isna().sum().sort_values(ascending=False)
print("\n[결측치 개수(내림차순)]")
print(na_cnt[na_cnt > 0])

# -----------------------------
# 3) 결측치 0으로 채우기
# -----------------변환------------
df = df.fillna(0)


# -----------------------------
# 4) X / y 분리
# -----------------------------
y = df["gender"]
X = df.drop(columns=["gender"])

X["환불금액"] = X["환불금액"].fillna(0)
X["환불여부"] = (X["환불금액"] > 0).astype(int)

for c in ["총구매액","최대구매액","환불금액"]:
    X[c] = np.log1p(X[c].clip(lower=0))


# -----------------------------
# 5) 문자형 범주 데이터 인코딩 (LabelEncoder)
#   - 주구매상품, 주구매지점 등 object 타입 컬럼 대상
# -----------------------------
# OneHotEncoding
cat_cols = ["주구매상품", "주구매지점"]
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print("\n[인코딩 적용한 범주형 컬럼]", cat_cols)

# -----------------------------
# 6) 표준화 (StandardScaler)
#   - 평균 0, 표준편차 1로 변환
# -----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nX_scaled shape:", X_scaled.shape)

# -----------------------------
# 7) 표준화 결과 확인(평균≈0, 표준편차≈1)
# -----------------------------
print("\n[표준화 후 각 컬럼 평균(근사)]")
print(np.round(X_scaled.mean(axis=0), 4))

print("\n[표준화 후 각 컬럼 표준편차(근사)]")
print(np.round(X_scaled.std(axis=0), 4))

Merged shape: (3500, 11)
   cust_id      총구매액     최대구매액       환불금액   주구매상품 주구매지점  내점일수   내점당구매건수  \
0        0  68282840  11264000  6860000.0      기타   강남점    19  3.894737   
1        1   2136000   2136000   300000.0     스포츠   잠실점     2  1.500000   
2        2   3197000   1639000        NaN  남성 캐주얼   관악점     2  2.000000   
3        3  16077620   4935000        NaN      기타   광주점    18  2.444444   
4        4  29050000  24000000        NaN      보석  본  점     2  1.500000   

     주말방문비율  구매주기  gender  
0  0.527027    17       0  
1  0.000000     1       0  
2  0.000000     1       1  
3  0.318182    16       1  
4  0.000000    85       0  

[결측치 개수(내림차순)]
환불금액    2295
dtype: int64

[인코딩 적용한 범주형 컬럼] ['주구매상품', '주구매지점']

X_scaled shape: (3500, 71)

[표준화 후 각 컬럼 평균(근사)]
[-0. -0.  0.  0. -0.  0. -0.  0.  0.  0.  0. -0.  0.  0. -0. -0. -0.  0.
 -0. -0.  0. -0. -0.  0. -0.  0.  0. -0. -0.  0. -0.  0. -0.  0.  0.  0.
  0.  0. -0. -0. -0. -0. -0.  0. -0. -0. -0. -0. -0.  0.  0.  0. -0. -0.
 -0.  0. 

# 5-1. 모델링 - LogisticRegression
- 본격적으로 모델을 선언하고 학습시킵니다.

필요한 라이브러리를 불러옵니다.

모델을 선언하여 객체화시킵니다.

모델을 학습 데이터에 맞춰 학습시킵니다.

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
# LogisticRegression에 class_weight 추가
lr_model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    class_weight="balanced",
    random_state=42
)
lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))

Test Accuracy: 0.6071428571428571


In [16]:
import pandas as pd

coef_df = pd.DataFrame({
    "feature": X.columns,
    "coef": lr_model.coef_[0]
}).sort_values("coef", ascending=False)

print(coef_df)

        feature      coef
55   주구매지점_본  점  0.377634
67    주구매지점_전주점  0.223247
66    주구매지점_잠실점  0.214340
58    주구매지점_분당점  0.206580
49    주구매지점_광주점  0.203861
..          ...       ...
68    주구매지점_창원점 -0.195261
47    주구매상품_화장품 -0.197756
41  주구매상품_침구/수예 -0.215262
28   주구매상품_시티웨어 -0.266639
17   주구매상품_디자이너 -0.314448

[71 rows x 2 columns]


# 6-1. 예측 성능 확인해보기 - LogisticRegression
- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.
- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [17]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

In [18]:
y_pred = lr_model.predict(X_test)

In [19]:
acc = accuracy_score(y_test, y_pred)
print("Accuracy :", acc)

Accuracy : 0.6071428571428571


In [20]:
precision = precision_score(y_test, y_pred)
print("Precision :", precision)

Precision : 0.48265895953757226


In [21]:
recall = recall_score(y_test, y_pred)
print("Recall :", recall)

Recall : 0.6349809885931559


In [22]:
f1 = f1_score(y_test, y_pred)
print("F1 Score :", f1)

F1 Score : 0.548440065681445


In [23]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix")
print(cm)

Confusion Matrix
[[258 179]
 [ 96 167]]


# 5-2. 모델링 - DecisionTreeClassifier
- 본격적으로 모델을 선언하고 학습시킵니다.

필요한 라이브러리를 불러옵니다.

모델을 선언하여 객체화시킵니다.

모델을 학습 데이터에 맞춰 학습시킵니다.

In [43]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, make_scorer
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(class_weight="balanced", random_state=42)

param_grid = {
    "max_depth": [3,4,5,6,7,8,10,12],
    "min_samples_leaf": [5,10,20,30,50],
    "min_samples_split": [2,10,20,40,80],
    "criterion": ["gini", "entropy"]
}

grid = GridSearchCV(
    dt,
    param_grid=param_grid,
    scoring=make_scorer(f1_score),  # 성능 목표를 F1로 잡기(불균형 대응)
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_dt = grid.best_estimator_
print("Best Params:", grid.best_params_)

Best Params: {'criterion': 'gini', 'max_depth': 4, 'min_samples_leaf': 30, 'min_samples_split': 2}


In [44]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_pred = best_dt.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1       :", f1_score(y_test, y_pred))
print("CM\n", confusion_matrix(y_test, y_pred))

Accuracy : 0.6042857142857143
Precision: 0.4782608695652174
Recall   : 0.5855513307984791
F1       : 0.5264957264957265
CM
 [[269 168]
 [109 154]]


# 6-2. 예측 성능 확인해보기 - DecisionTreeClassifier

- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.
- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [46]:
# 평가
y_pred = best_dt.predict(X_test)
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1       :", f1_score(y_test, y_pred))
print("Confusion Matrix\n", confusion_matrix(y_test, y_pred))

Accuracy : 0.6042857142857143
Precision: 0.4782608695652174
Recall   : 0.5855513307984791
F1       : 0.5264957264957265
Confusion Matrix
 [[269 168]
 [109 154]]


# 5-3. 모델링 - RandomForestClassifier
- 본격적으로 모델을 선언하고 학습시킵니다.

필요한 라이브러리를 불러옵니다.

모델을 선언하여 객체화시킵니다.

모델을 학습 데이터에 맞춰 학습시킵니다.

In [49]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

param_dist = {
    "n_estimators": [300, 500, 800, 1200],
    "max_depth": [None, 5, 8, 12, 16, 20],
    "min_samples_split": [2, 10, 20, 40],
    "min_samples_leaf": [1, 5, 10, 20, 30],
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False]
}

search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=40,
    scoring=make_scorer(f1_score),  # F1 기준 최적화
    cv=5,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)
best_rf = search.best_estimator_

print("Best Params:", search.best_params_)

Best Params: {'n_estimators': 300, 'min_samples_split': 40, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 12, 'bootstrap': False}


# 6-3. 예측 성능 확인해보기 - RandomForestClassifier
- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.
- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [50]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_pred = best_rf.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1       :", f1_score(y_test, y_pred))
print("CM\n", confusion_matrix(y_test, y_pred))

Accuracy : 0.6128571428571429
Precision: 0.48773006134969327
Recall   : 0.6045627376425855
F1       : 0.5398981324278438
CM
 [[270 167]
 [104 159]]


# 5-4. 모델링 - XGBoost
- 본격적으로 모델을 선언하고 학습시킵니다.

필요한 라이브러리를 불러옵니다.

모델을 선언하여 객체화시킵니다.

모델을 학습 데이터에 맞춰 학습시킵니다.

In [52]:
from xgboost import XGBClassifier

In [53]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
scale_pos_weight

np.float64(1.6590693257359923)

In [57]:
xgb_model = XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=3,
    gamma=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

In [58]:
xgb_model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [59]:
y_pred = xgb_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1       :", f1_score(y_test, y_pred))
print("Confusion Matrix\n", confusion_matrix(y_test, y_pred))

Accuracy : 0.6071428571428571
Precision: 0.47959183673469385
Recall   : 0.5361216730038023
F1       : 0.5062836624775583
Confusion Matrix
 [[284 153]
 [122 141]]
